# MTG-I (Meteosat Third Generation Imager) / LI (Lightning Imager)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=mtg/mtg-li). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L2

Level 2: lightning events, groups and flashes (LFL, LGR), and accumulated products on a regular grid (AF accumulated flashes, AFA accumulated flash area, AFR accumulated flash radiance).

Downloads one LI accumulated-flashes product (EO:EUM:DAT:0686) from the EUMETSAT Data Store (needs a free account and consumer key/secret, asked at run time) and plots the first dataset with satpy.

In [ ]:
%pip install -q eumdac satpy netCDF4 h5netcdf matplotlib

In [ ]:
import datetime as dt
import getpass
import shutil
import zipfile
from pathlib import Path

import eumdac

# Free consumer key / secret: https://api.eumetsat.int/api-key (EUMETSAT Data Store account)
token = eumdac.AccessToken((input("EUMETSAT consumer key: "), getpass.getpass("EUMETSAT consumer secret: ")))
datastore = eumdac.DataStore(token)
collection = datastore.get_collection("EO:EUM:DAT:0686")

products = list(collection.search(dtstart=dt.datetime(2025, 6, 1, 15, 0), dtend=dt.datetime(2025, 6, 1, 15, 10)))
print(len(products), "LI accumulated flashes products found")
if not products:
    raise SystemExit("Nothing in this time window - try another one")
product = products[0]
print(product)

out = Path("data")
out.mkdir(exist_ok=True)
# a few MB per product
with product.open() as src, open(out / src.name, "wb") as dst:
    shutil.copyfileobj(src, dst)
for z in out.glob("*.zip"):                     # some collections arrive zipped
    with zipfile.ZipFile(z) as zf:
        zf.extractall(out / z.stem)

from glob import glob

import matplotlib.pyplot as plt
from satpy import Scene

files = sorted(glob("data/**/*.nc", recursive=True))
print(len(files), "files")
scn = Scene(filenames=files, reader="li_l2_nc")
print(scn.available_dataset_names())
name = scn.available_dataset_names()[0]
scn.load([name])
scn[name].plot()
plt.title(name)
plt.show()